In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, to_hetero
from torch_geometric.data import DataLoader
from halide_gnn_cost_model.data import PipelineDataset
from pathlib import Path

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
dataset = PipelineDataset(Path("../pipelines"))
data = dataset[0]  # Get the first pipeline graph
print(data.metadata)

<bound method HeteroData.metadata of HeteroData(
  y=[5],
  function={ x=[8, 1] },
  (function, called_by, function)={ edge_index=[2, 8] }
)>


In [3]:
data

HeteroData(
  y=[5],
  function={ x=[8, 1] },
  (function, called_by, function)={ edge_index=[2, 8] }
)

In [4]:
class PipeGCN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers=2):
        super(PipeGCN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(-1, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(-1, hidden_channels))
        self.convs.append(GCNConv(-1, out_channels))

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x

In [5]:
gcn = PipeGCN(hidden_channels=32, out_channels=32, num_layers=3)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")

/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/.venv/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()


In [6]:
out = gcn(data.x_dict, data.edge_index_dict)
out["function"][0]

tensor([ 0.0090, -0.0242,  0.3047, -0.0599,  0.0540, -0.1133, -0.1977,  0.0010,
         0.1417, -0.1877,  0.1859, -0.0441, -0.1825,  0.1487, -0.0374,  0.0291,
        -0.0332, -0.1485,  0.0042,  0.1574,  0.0143,  0.2514, -0.0503, -0.0915,
        -0.2839, -0.0279,  0.1104, -0.0072,  0.1390,  0.0895,  0.0174, -0.1641],
       grad_fn=<SelectBackward0>)

In [7]:
class PipelineModel(torch.nn.Module):
    def __init__(self, gnn, out_channels, num_runtime):
        super(PipelineModel, self).__init__()
        self.function_gnn = gnn
        self.pipeline_lin = torch.nn.Linear(out_channels, num_runtime)

    def forward(self, data, ptr=None):
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict
        out = self.function_gnn(x_dict, edge_index_dict)
        # Get the feature of the pipeline node
        idx = 0 if ptr is None else ptr
        pipeline_feat = out["function"][idx]
        # Predict the runtime
        x = self.pipeline_lin(pipeline_feat)
        run_time = torch.exp(x)
        return run_time

In [8]:
model = PipelineModel(gcn, 32, 5)
model

PipelineModel(
  (function_gnn): GraphModule(
    (convs): ModuleList(
      (0-2): 3 x ModuleDict(
        (function__called_by__function): GCNConv(-1, 32)
      )
    )
  )
  (pipeline_lin): Linear(in_features=32, out_features=5, bias=True)
)

In [9]:
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)
data_loader

/var/folders/qh/c7l883gn5s548l5w18l25cwc0000gn/T/nix-shell.e8I3Oz/ipykernel_83803/1643914774.py:1: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  data_loader = DataLoader(dataset, batch_size=4, shuffle=True)


In [10]:
# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.L1Loss()

for epoch in range(1000):
    model.train()
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        pred = model(batch, batch["function"].ptr[:-1])
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(data_loader)}")

Epoch 100, Loss: 152.3208236694336
Epoch 200, Loss: 146.01014709472656
Epoch 300, Loss: 137.69759511947632
Epoch 400, Loss: 132.81711196899414
Epoch 500, Loss: 139.6784634590149
Epoch 600, Loss: 134.50639581680298
Epoch 700, Loss: 127.46575736999512
Epoch 800, Loss: 132.58054518699646
Epoch 900, Loss: 131.65883231163025
Epoch 1000, Loss: 129.1855595111847


In [11]:
torch.set_printoptions(precision=4)
print(model(data))

tensor([  0.5253,   2.2353,   9.2936,  40.5462, 227.6091],
       grad_fn=<ExpBackward0>)
